In [1]:
# ============================================================
# CELL 1 — Imports and setup
# ============================================================
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
from IPython.display import display
import os
import sys
sys.path.append('..')

# Publication style
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

CITIES = ['manhattan', 'pittsburgh', 'philadelphia']
CITY_COLORS = {'manhattan': '#2196F3', 'pittsburgh': '#FF9800', 'philadelphia': '#4CAF50'}
LABEL_COLORS = {'Answerable': '#4CAF50', 'Ambiguous': '#FF9800', 'Contradictory': '#F44336'}
VARIANT_COLORS = {'mask_landmark': '#9C27B0', 'mask_directions': '#2196F3', 'mask_both': '#FF5722'}

REPORTS_DIR = '../reports/llm_audits'
DATA_DIR = '../data'

os.makedirs(f'{REPORTS_DIR}/figures', exist_ok=True)
print("✅ Setup complete")

✅ Setup complete


In [2]:
# ============================================================
# NOTEBOOK 2 — CELL 1: Load and validate LLM_DEGRADATION_INPUT
# ============================================================
input_df = pd.read_parquet(f'{REPORTS_DIR}/LLM_DEGRADATION_INPUT.parquet')
print(f"✅ Loaded LLM_DEGRADATION_INPUT: {len(input_df)} rows × {len(input_df.columns)} columns")
print(f"\nColumns: {input_df.columns.tolist()}")
print(f"\nCities: {input_df['city'].value_counts().to_dict()}")
print(f"Variant types: {input_df['variant_type'].value_counts().to_dict()}")
print(f"Oracle labels: {input_df['oracle_label'].value_counts().to_dict()}")
print(f"\nNull values:")
display(input_df.isnull().sum()[input_df.isnull().sum() > 0])

✅ Loaded LLM_DEGRADATION_INPUT: 21049 rows × 15 columns

Columns: ['sample_id', 'city', 'original_text', 'masked_instruction', 'variant_type', 'removed_element', 'extracted_category', 'extracted_direction', 'extracted_noun', 'start_node', 'gold_goal_node', 'gold_goal_lat', 'gold_goal_lon', 'oracle_label', 'reachable_candidate_count']

Cities: {'manhattan': 16066, 'philadelphia': 2942, 'pittsburgh': 2041}
Variant types: {'mask_landmark': 7163, 'mask_directions': 6943, 'mask_both': 6943}
Oracle labels: {'Contradictory': 9030, 'Ambiguous': 8169, 'Answerable': 3850}

Null values:


extracted_direction    220
dtype: int64

In [3]:
# ============================================================
# NOTEBOOK 2 — CELL 2: Input file deep validation
# ============================================================
print("=== LLM_DEGRADATION_INPUT Validation ===\n")
issues = []

# 1. Duplicate rows
dupes = input_df.duplicated().sum()
print(f"1. Duplicate rows: {dupes}")
if dupes > 0: issues.append(f"{dupes} duplicate rows")

# 2. Null gold coordinates
lat_nulls = input_df['gold_goal_lat'].isna().sum()
lon_nulls = input_df['gold_goal_lon'].isna().sum()
print(f"2. gold_goal_lat nulls: {lat_nulls} | gold_goal_lon nulls: {lon_nulls}")
if lat_nulls > 0: issues.append(f"{lat_nulls} null coordinates")

# 3. Null oracle labels
label_nulls = input_df['oracle_label'].isna().sum()
print(f"3. Null oracle labels: {label_nulls}")

# 4. Null start_node
node_nulls = input_df['start_node'].isna().sum()
print(f"4. Null start_node: {node_nulls}")

# 5. Valid oracle label values
valid_labels = {'Answerable', 'Ambiguous', 'Contradictory'}
invalid = ~input_df['oracle_label'].isin(valid_labels)
print(f"5. Invalid oracle labels: {invalid.sum()}")

# 6. Coordinate range sanity
lat_range = input_df['gold_goal_lat'].agg(['min','max'])
lon_range = input_df['gold_goal_lon'].agg(['min','max'])
print(f"6. Lat range: {lat_range['min']:.4f} to {lat_range['max']:.4f}")
print(f"   Lon range: {lon_range['min']:.4f} to {lon_range['max']:.4f}")

# 7. mask_both <= mask_directions per city
for city in input_df['city'].unique():
    city_df = input_df[input_df['city'] == city]
    md = (city_df['variant_type'] == 'mask_directions').sum()
    mb = (city_df['variant_type'] == 'mask_both').sum()
    status = "✅" if md >= mb else "❌"
    print(f"7. {city}: mask_directions({md}) >= mask_both({mb}) {status}")

print(f"\n{'✅ All checks passed' if not issues else '⚠️  Issues: ' + str(issues)}")

=== LLM_DEGRADATION_INPUT Validation ===

1. Duplicate rows: 0
2. gold_goal_lat nulls: 0 | gold_goal_lon nulls: 0
3. Null oracle labels: 0
4. Null start_node: 0
5. Invalid oracle labels: 0
6. Lat range: 39.9140 to 40.7857
   Lon range: -80.0347 to -73.9493
7. manhattan: mask_directions(5299) >= mask_both(5299) ✅
7. pittsburgh: mask_directions(668) >= mask_both(668) ✅
7. philadelphia: mask_directions(976) >= mask_both(976) ✅

✅ All checks passed


In [4]:
# ============================================================
# NOTEBOOK 2 — CELL 3: Load and validate LLM_DEGRADATION_RESULTS
# ============================================================
results_path = f'{REPORTS_DIR}/LLM_DEGRADATION_RESULTS.parquet'
if not os.path.exists(results_path):
    print("❌ Results file not found — run evaluate_llm_masked.py first")
else:
    results_df = pd.read_parquet(results_path)
    print(f"✅ Loaded results: {len(results_df)} rows × {len(results_df.columns)} columns")
    print(f"\nColumns: {results_df.columns.tolist()}")
    display(results_df.dtypes)

✅ Loaded results: 50 rows × 15 columns

Columns: ['sample_id', 'city', 'variant_type', 'oracle_label', 'extracted_category', 'masked_instruction', 'llm_output_raw', 'resolution_succeeded', 'predicted_lat', 'predicted_lon', 'gold_goal_lat', 'gold_goal_lon', 'distance_m', 'success_250m', 'success_100m']


sample_id                 int64
city                     object
variant_type             object
oracle_label             object
extracted_category       object
masked_instruction       object
llm_output_raw           object
resolution_succeeded       bool
predicted_lat           float64
predicted_lon           float64
gold_goal_lat           float64
gold_goal_lon           float64
distance_m              float64
success_250m             object
success_100m             object
dtype: object

In [5]:
# ============================================================
# NOTEBOOK 2 — CELL 4: Results file validation
# ============================================================
print("=== LLM_DEGRADATION_RESULTS Validation ===\n")

# 1. Resolution rate
resolved = results_df['resolution_succeeded'].sum()
total = len(results_df)
print(f"1. Resolution rate: {resolved}/{total} = {resolved/total:.1%}")

# 2. Resolution rate by oracle label
print(f"\n2. Resolution rate by oracle label:")
display(results_df.groupby('oracle_label')['resolution_succeeded'].mean().round(3))

# 3. Success metric nulls
s250_nulls = results_df['success_250m'].isna().sum()
s100_nulls = results_df['success_100m'].isna().sum()
print(f"\n3. success_250m nulls: {s250_nulls} ({s250_nulls/total:.1%})")
print(f"   success_100m nulls: {s100_nulls} ({s100_nulls/total:.1%})")
print(f"   (nulls = unresolved predictions — expected)")

# 4. Null llm output
empty_output = (results_df['llm_output_raw'].str.strip() == '').sum()
null_output = results_df['llm_output_raw'].isna().sum()
print(f"\n4. Empty LLM outputs: {empty_output} | Null: {null_output}")

# 5. Sample good rows
print(f"\n5. Sample RESOLVED rows:")
display(results_df[results_df['resolution_succeeded']][
    ['variant_type','oracle_label','llm_output_raw','distance_m','success_250m']
].head(5))

# 6. Sample unresolved rows
print(f"\n6. Sample UNRESOLVED rows:")
display(results_df[~results_df['resolution_succeeded']][
    ['variant_type','oracle_label','llm_output_raw']
].head(5))

# 7. Distance distribution for resolved
resolved_df = results_df[results_df['distance_m'].notna()]
print(f"\n7. Distance distribution (resolved only):")
print(resolved_df['distance_m'].describe().round(1))

=== LLM_DEGRADATION_RESULTS Validation ===

1. Resolution rate: 14/50 = 28.0%

2. Resolution rate by oracle label:


oracle_label
Ambiguous        0.280
Answerable       0.333
Contradictory    0.231
Name: resolution_succeeded, dtype: float64


3. success_250m nulls: 36 (72.0%)
   success_100m nulls: 36 (72.0%)
   (nulls = unresolved predictions — expected)

4. Empty LLM outputs: 0 | Null: 0

5. Sample RESOLVED rows:


,variant_type,oracle_label,llm_output_raw,distance_m,success_250m
3,mask_landmark,Answerable,The United Nations,1184.753453,False
4,mask_directions,Ambiguous,United Nations,1184.753453,False
5,mask_both,Answerable,United Nations,1184.753453,False
15,mask_both,Contradictory,10th Avenue,698.616357,False
17,mask_directions,Ambiguous,Gotham Pizza,79.924797,True



6. Sample UNRESOLVED rows:


,variant_type,oracle_label,llm_output_raw
0,mask_landmark,Ambiguous,Liberty Street
1,mask_directions,Ambiguous,Liberty Street
2,mask_both,Contradictory,Liberty Street
6,mask_landmark,Answerable,[MASK]
7,mask_directions,Ambiguous,Creperie



7. Distance distribution (resolved only):
count      14.0
mean      682.4
std       571.2
min        22.2
25%        79.9
50%       663.3
75%      1150.3
max      1765.7
Name: distance_m, dtype: float64
